# Step 5C: 횡단면 랭크 기반 상대뷰 (Relative Views)

## 개념: 절대뷰 vs 상대뷰

### Step5 기본 (절대뷰, P = Identity)
```
"AAPL은 연 30% 수익률을 낼 것이다"
P = I, Q = [Q_AAPL, Q_MSFT, ...]
```

### Step5C (상대뷰, 쌍 비교)
```
"AAPL이 MSFT보다 +Δ% 더 수익을 낼 것이다"
P_ij = [+1 (long AAPL), -1 (short MSFT), 0, 0, ...]
Q_ij = rank_normalized_momentum_AAPL - rank_normalized_momentum_MSFT
```

## 장점
- 절대 수익률 추정의 노이즈 감소
- BL 원래 논문(He & Litterman 1992)의 실제 사용 방식
- 같은 섹터 내 종목 비교가 특히 효과적

## 구현 전략: 섹터 내 상위-하위 쌍 비교
```
각 섹터에서:
  - 모멘텀 랭크 1위 vs 5위 → "1위가 5위보다 α% 더 좋을 것"
  - 뷰 수 = 섹터당 2쌍 = 11 × 2 = 22개 뷰
```

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
import warnings
import pickle
import yfinance as yf

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

BASE = Path('.')
DATA = BASE / 'data'
IMAGES = BASE / 'images'
CACHE = DATA / 'stock5_cache'

TOP_N = 5
REBALANCE_FREQ = 21
UNIVERSE_UPDATE_FREQ = 252
ANALYSIS_START = '2016-01-01'
PRICE_START = '2013-01-01'
PRICE_END = '2025-12-31'
COV_WIN = 252
TAU = 1 / 252
LAM = 2.5

print('Step 5C: 횡단면 상대뷰 — 준비 완료')

In [ ]:
import io, requests

def fetch_sp500_snapshot(cache_path=None):
    cache_path = cache_path or (CACHE / 'sp500_snapshot.pkl')
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            return pickle.load(f)
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; research-bot)'}
    resp = requests.get(url, headers=headers, timeout=30)
    tables = pd.read_html(io.StringIO(resp.text))
    df = tables[0].copy()
    df.columns = [str(c).strip() for c in df.columns]
    ticker_col = next(c for c in df.columns if 'symbol' in c.lower() or 'ticker' in c.lower())
    sector_col = next(
        c for c in df.columns
        if 'gics sector' in c.lower() or ('sector' in c.lower() and 'sub' not in c.lower())
    )
    result = df[[ticker_col, sector_col]].rename(
        columns={ticker_col: 'ticker', sector_col: 'sector'}
    ).copy()
    result['ticker'] = result['ticker'].str.replace('.', '-', regex=False).str.strip()
    result = result.dropna().reset_index(drop=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(result, f)
    print(f'S&P 500 종목 {len(result)}개 수집')
    return result


def fetch_sp500_changes(cache_path=None):
    cache_path = cache_path or (CACHE / 'sp500_changes.pkl')
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            return pickle.load(f)
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; research-bot)'}
    resp = requests.get(url, headers=headers, timeout=30)
    tables = pd.read_html(io.StringIO(resp.text))
    raw = tables[1].copy()
    raw.columns = [str(c).strip() for c in raw.columns]
    date_col = raw.columns[0]
    def find_col(df, keywords):
        for c in df.columns:
            if any(k.lower() in str(c).lower() for k in keywords): return c
        return None
    added_col   = find_col(raw, ['added', 'add']) or raw.columns[1]
    removed_col = find_col(raw, ['removed', 'remove']) or raw.columns[3]
    changes = pd.DataFrame({
        'date':    pd.to_datetime(raw[date_col], errors='coerce'),
        'added':   raw[added_col].astype(str).str.strip().str.replace('.', '-', regex=False),
        'removed': raw[removed_col].astype(str).str.strip().str.replace('.', '-', regex=False),
    }).dropna(subset=['date'])
    changes['added']   = changes['added'].replace({'nan': None, 'NaN': None, '': None})
    changes['removed'] = changes['removed'].replace({'nan': None, 'NaN': None, '': None})
    changes = changes.sort_values('date').reset_index(drop=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(changes, f)
    return changes


def filter_pool_by_date(snapshot_df, changes_df, target_date):
    target_dt   = pd.Timestamp(target_date)
    added_after = set(
        changes_df.loc[
            (changes_df['date'] > target_dt) & changes_df['added'].notna(), 'added'
        ].dropna()
    )
    filtered_df = snapshot_df[~snapshot_df['ticker'].isin(added_after)].reset_index(drop=True)
    excluded    = sorted(set(snapshot_df['ticker']) & added_after)
    return filtered_df, excluded


sp500_snapshot = fetch_sp500_snapshot()
sp500_changes  = fetch_sp500_changes()
print(f'후보 풀: {len(sp500_snapshot)}개 종목, {sp500_snapshot["sector"].nunique()}개 섹터')


# ── 채권/대안 ETF (통합 BL) ──────────────────────────────────────────────
# Asness et al. 2013: 모멘텀은 주식·채권·원자재 모두 동일 공식 적용 가능
BOND_TICKERS = ['TLT', 'SHY', 'TIP', 'AGG', 'GLD', 'DBC']

bond_cache = CACHE / 'bond_prices.pkl'
if bond_cache.exists():
    with open(bond_cache, 'rb') as f:
        bond_prices = pickle.load(f)
    print(f'채권 데이터 로드: {bond_prices.shape} (캐시)')
else:
    print('채권 ETF 데이터 다운로드 중...')
    raw = yf.download(BOND_TICKERS, start=PRICE_START, end=PRICE_END,
                      auto_adjust=True)['Close']
    if isinstance(raw, pd.Series):
        raw = raw.to_frame()
    bond_prices = raw[BOND_TICKERS].dropna(how='all').ffill()
    with open(bond_cache, 'wb') as f:
        pickle.dump(bond_prices, f)
    print(f'채권 데이터 저장: {bond_prices.shape}')

# ── prices_combined ───────────────────────────────────────────────────────
prices_combined = prices.copy()
for bt in BOND_TICKERS:
    if bt in bond_prices.columns:
        prices_combined[bt] = bond_prices[bt]
prices_combined = prices_combined.ffill()
print(f'통합 가격 데이터: {prices_combined.shape}')


## 셀 3: 상대뷰 핵심 함수

P 행렬 (n_views × n_assets): 각 행이 하나의 상대뷰
- `+1`: long 종목 (더 좋을 것으로 예상)
- `-1`: short 종목 (상대적으로 나쁠 것으로 예상)

In [ ]:
def compute_momentum_scores(prices_window, windows=(126, 189, 252)):
    """종목별 모멘텀 점수 (연환산 로그 수익률 평균)"""
    log_p = np.log(prices_window)
    qs = []
    for w in windows:
        if len(log_p) > w:
            qs.append((log_p.iloc[-1] - log_p.iloc[-(w+1)]) / w * 252)
    return pd.concat(qs, axis=1).mean(axis=1) if qs else pd.Series(0.0, index=prices_window.columns)


def build_relative_views(momentum_scores, sector_map, top_k=2):
    """
    섹터 내 상위-하위 쌍 비교로 상대뷰 생성

    Parameters
    ----------
    momentum_scores : pd.Series - 종목별 모멘텀 점수
    sector_map : dict           - 섹터 → 종목 리스트
    top_k : int                 - 섹터당 쌍 수 (상위 k vs 하위 k)

    Returns
    -------
    P : np.ndarray  - (n_views × n_assets) 뷰 행렬
    q : np.ndarray  - (n_views,) 뷰 수익률
    sigma_v : np.ndarray - (n_views,) 뷰 신뢰도 (표준편차)
    view_labels : list  - 뷰 설명 텍스트
    """
    all_tickers = list(momentum_scores.index)
    n_assets = len(all_tickers)
    ticker_idx = {t: i for i, t in enumerate(all_tickers)}

    P_rows, q_vals, sigma_vals, labels = [], [], [], []

    # 전체 횡단면 랭크 (0~1 정규화)
    rank_normalized = momentum_scores.rank(pct=True)  # 0~1

    for sector, tickers in sector_map.items():
        if len(tickers) < 2:
            continue

        # 섹터 내 모멘텀 순위
        sector_scores = momentum_scores.reindex(tickers).dropna().sort_values(ascending=False)
        n_s = len(sector_scores)
        if n_s < 2:
            continue

        # Top vs Bottom 쌍 (top_k개)
        n_pairs = min(top_k, n_s // 2)
        for k in range(n_pairs):
            long_ticker = sector_scores.index[k]       # 순위 k+1
            short_ticker = sector_scores.index[-(k+1)] # 순위 하위 k+1

            p_row = np.zeros(n_assets)
            p_row[ticker_idx[long_ticker]] = 1.0
            p_row[ticker_idx[short_ticker]] = -1.0
            P_rows.append(p_row)

            # 뷰 수익률: 랭크 차이를 수익률 차이로 스케일
            rank_diff = rank_normalized[long_ticker] - rank_normalized[short_ticker]
            q_view = rank_diff * 0.20  # 스케일: 최대 랭크 차이(1) → 연 20% 차이
            q_vals.append(q_view)

            # 불확실성: 쌍 간 랭크 차이가 클수록 더 확신
            confidence = 0.10 + 0.80 * rank_diff
            omega_view = ((1 - confidence) / confidence) * TAU * 0.04  # 연간 표준편차 20% 기준
            sigma_vals.append(omega_view)

            labels.append(f'{sector[:15]}: {long_ticker}>{short_ticker} ({rank_diff:.2f})')

    if not P_rows:
        return None, None, None, []

    return np.array(P_rows), np.array(q_vals), np.array(sigma_vals), labels


def black_litterman_relative(pi, cov, P, q, omega_diag, tau=1/252):
    """
    상대뷰 BL: P 행렬이 Identity가 아닌 경우
    μ_BL = [(τΣ)⁻¹ + P'Ω⁻¹P]⁻¹ × [(τΣ)⁻¹π + P'Ω⁻¹q]
    """
    n = len(pi)
    n_views = len(q)
    omega = np.diag(omega_diag)

    tau_cov_inv = np.linalg.inv(tau * cov + np.eye(n) * 1e-8)
    omega_inv = np.linalg.inv(omega + np.eye(n_views) * 1e-10)

    precision = tau_cov_inv + P.T @ omega_inv @ P
    rhs = tau_cov_inv @ pi.values + P.T @ omega_inv @ q
    mu_bl = np.linalg.solve(precision + np.eye(n) * 1e-8, rhs)
    sigma_bl = cov + np.linalg.inv(precision + np.eye(n) * 1e-8)
    return pd.Series(mu_bl, index=pi.index), sigma_bl


# 단일 기간 테스트
latest_pool, _ = filter_pool_by_date(sp500_snapshot, sp500_changes, prices.index[-1])
test_univ, test_sm, test_wm = select_universe_by_mcap(
    prices.iloc[-1], shares_dict, latest_pool, TOP_N
)
test_window = prices.loc[:prices.index[-1], test_univ]
mom_scores = compute_momentum_scores(test_window)
P_test, q_test, sig_test, labels_test = build_relative_views(mom_scores, test_sm, top_k=2)

print(f'생성된 상대뷰 수: {len(q_test) if q_test is not None else 0}')
if labels_test:
    print('\n뷰 내용 (상위 10개):')
    for i, (lbl, qv) in enumerate(zip(labels_test[:10], q_test[:10])):
        print(f'  {i+1:2d}. {lbl:45s}: Q={qv:+.4f}')

## 셀 4: 공통 함수 재정의

In [ ]:
from scipy.optimize import minimize

def compute_prior_risk_parity(cov, lam=2.5):
    """Risk Parity Prior (Maillard et al. 2010)"""
    n  = cov.shape[0]
    s  = np.sqrt(np.diag(cov))
    w0 = (1.0 / s) / (1.0 / s).sum()
    def erc_obj(w):
        pv = float(w @ cov @ w)
        if pv < 1e-12: return 1e10
        rc = w * (cov @ w) / np.sqrt(pv)
        return float(np.sum((rc - np.sqrt(pv)/n)**2))
    res = minimize(erc_obj, w0, method='SLSQP',
                   bounds=[(1e-4, 1.0)]*n,
                   constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-1}],
                   options={'ftol': 1e-10, 'maxiter': 1000})
    w_rp = np.clip(res.x if res.success else w0, 0, 1)
    w_rp /= w_rp.sum()
    return pd.Series(lam * cov @ w_rp, index=range(n))


def select_universe_by_mcap(price_snap, shares_dict, pool_df, top_n=5):
    sector_map = {}
    mcap_map   = {}
    for sector, grp in pool_df.groupby('sector'):
        avail = [
            t for t in grp['ticker']
            if t in price_snap.index
            and pd.notna(price_snap.get(t, np.nan))
            and shares_dict.get(t, 0) > 0
        ]
        if not avail: continue
        mcap     = {t: price_snap[t] * shares_dict[t] for t in avail}
        selected = sorted(mcap.items(), key=lambda x: x[1], reverse=True)[:top_n]
        sector_map[sector] = [t for t, _ in selected]
        for t, m in selected: mcap_map[t] = m
    universe   = [t for ts in sector_map.values() for t in ts]
    total_mcap = sum(mcap_map.values()) or 1
    w_mkt      = pd.Series({t: mcap_map[t] / total_mcap for t in universe})
    return universe, sector_map, w_mkt


def mvo_optimize_unified(mu, cov, gamma, equity_tickers, bond_tickers,
                          max_equity_weight, max_sector, sector_map,
                          max_bond_total, max_single_bond):
    """통합 MVO: 주식 + 채권/대안 ETF (프로파일별 비중 제약)"""
    all_tickers = list(equity_tickers) + list(bond_tickers)
    n = len(all_tickers)
    mu_arr  = mu.reindex(all_tickers).fillna(0).values
    eq_idx  = list(range(len(equity_tickers)))
    bnd_idx = list(range(len(equity_tickers), n))

    def neg_utility(w):
        return -(w @ mu_arr - (gamma / 2) * w @ cov @ w)

    constraints = [
        {'type': 'eq',   'fun': lambda w: w.sum() - 1.0},
        {'type': 'ineq', 'fun': lambda w: max_bond_total - w[bnd_idx].sum()},
    ]
    if sector_map:
        for sector, s_tickers in sector_map.items():
            idx = [eq_idx[list(equity_tickers).index(t)]
                   for t in s_tickers if t in list(equity_tickers)]
            if idx:
                constraints.append({
                    'type': 'ineq',
                    'fun': lambda w, i=idx: max_sector - sum(w[j] for j in i)
                })

    bounds = ([(0.0, max_equity_weight)] * len(equity_tickers) +
              [(0.0, max_single_bond)]   * len(bond_tickers))

    res = minimize(neg_utility, np.ones(n) / n, method='SLSQP',
                   bounds=bounds, constraints=constraints,
                   options={'ftol': 1e-9, 'maxiter': 1000})
    w = np.clip(res.x if res.success else np.ones(n)/n, 0, 1)
    return pd.Series(w / w.sum(), index=all_tickers)


# ── 리스크 프로파일 (채권 비중 제약 포함) ─────────────────────────────────
PROFILES = {
    'Aggressive':   {'gamma': 2, 'max_equity_weight': 0.15, 'max_sector': 0.50,
                     'max_bond_total': 0.20, 'max_single_bond': 0.12},
    'Neutral':      {'gamma': 5, 'max_equity_weight': 0.08, 'max_sector': 0.35,
                     'max_bond_total': 0.35, 'max_single_bond': 0.12},
    'Conservative': {'gamma': 8, 'max_equity_weight': 0.05, 'max_sector': 0.25,
                     'max_bond_total': 0.55, 'max_single_bond': 0.20},
}

print('공통 함수 정의 완료 (Risk Parity Prior + 통합 MVO)')
print(f'{"프로파일":15s} {"γ":>4s} {"max_eq_w":>9s} {"max_bond":>9s}')
for name, p in PROFILES.items():
    print(f'{name:15s} {p["gamma"]:>4} {p["max_equity_weight"]:>9.0%} {p["max_bond_total"]:>9.0%}')


## 셀 5: Walk-Forward 백테스트 (상대뷰)

In [ ]:
analysis_days = prices_combined.loc[ANALYSIS_START:].index

# 초기 유니버스
start_snap = prices.loc[analysis_days[0]]
init_pool, _ = filter_pool_by_date(sp500_snapshot, sp500_changes, analysis_days[0])
current_eq_univ, current_sector_map, _ = select_universe_by_mcap(
    start_snap, shares_dict, init_pool, TOP_N
)

def build_combined_universe(eq_univ, bond_tickers):
    return list(eq_univ) + [b for b in bond_tickers if b not in eq_univ]

current_universe = build_combined_universe(current_eq_univ, BOND_TICKERS)
n_all = len(current_universe)

current_weights = {p: pd.Series(1.0/n_all, index=current_universe) for p in PROFILES}
port_values     = {p: [1.0] for p in PROFILES}

print(f'분석 기간: {analysis_days[0].date()} ~ {analysis_days[-1].date()}')
print(f'전략: 섹터 내 상위-하위 쌍 상대뷰 BL (주식 상대뷰 + 채권 Prior 포함)')
print(f'초기 유니버스: 주식 {len(current_eq_univ)} + 채권/대안 {len(BOND_TICKERS)} = {n_all}개\n')

for i, date in enumerate(analysis_days[1:], 1):
    prev_date = analysis_days[i-1]

    ret = {}
    for t in current_universe:
        if t in prices_combined.columns:
            p0 = prices_combined.at[prev_date, t] if prev_date in prices_combined.index else np.nan
            p1 = prices_combined.at[date, t]      if date      in prices_combined.index else np.nan
            ret[t] = (p1/p0 - 1) if (pd.notna(p0) and pd.notna(p1) and p0 > 0) else 0.0
        else:
            ret[t] = 0.0

    for p in PROFILES:
        w = current_weights[p]
        port_ret = sum(w.get(t, 0) * ret.get(t, 0) for t in current_universe)
        port_values[p].append(port_values[p][-1] * (1 + port_ret))

    # 연간 유니버스 갱신
    if i % UNIVERSE_UPDATE_FREQ == 0:
        year_pool, _ = filter_pool_by_date(sp500_snapshot, sp500_changes, date)
        new_eq_univ, new_sector_map, _ = select_universe_by_mcap(
            prices.loc[date], shares_dict, year_pool, TOP_N
        )
        current_eq_univ    = new_eq_univ
        current_sector_map = new_sector_map
        current_universe   = build_combined_universe(current_eq_univ, BOND_TICKERS)
        n_all = len(current_universe)
        for p in PROFILES:
            current_weights[p] = pd.Series(1.0/n_all, index=current_universe)
        print(f'  [{date.date()}] 유니버스 갱신: 주식 {len(current_eq_univ)}+채권 {len(BOND_TICKERS)} = {n_all}개')

    # 21일마다 BL 리밸런싱
    if i % REBALANCE_FREQ == 0:
        prices_win = prices_combined.loc[:date, current_universe]
        if len(prices_win) < COV_WIN + 253:
            continue

        recent = prices_win.iloc[-COV_WIN:]
        log_r  = np.log(recent / recent.shift(1)).dropna()
        if log_r.shape[0] < 60:
            continue

        lw = LedoitWolf()
        lw.fit(log_r)
        cov = lw.covariance_ * 252

        # Risk Parity Prior (전 자산)
        pi_raw = compute_prior_risk_parity(cov, LAM)
        pi = pd.Series(pi_raw.values, index=current_universe)

        # 상대뷰: 주식 섹터 내에서만 생성
        # (채권/대안 ETF는 P 행렬에 포함 안 됨 → Prior 그대로 반영)
        eq_prices_win = prices_win[list(current_eq_univ)]
        mom_scores = compute_momentum_scores(eq_prices_win)
        P_mat, q_vec, sigma_vec, view_labels = build_relative_views(
            mom_scores, current_sector_map, top_k=2
        )

        if P_mat is None or len(q_vec) == 0:
            mu_bl    = pi
            sigma_bl = cov
        else:
            # P 행렬을 전체 유니버스 크기로 확장 (채권 열 = 0)
            n_eq   = len(current_eq_univ)
            n_bond = len(BOND_TICKERS)
            n_all_curr = n_eq + n_bond
            P_full = np.hstack([P_mat, np.zeros((P_mat.shape[0], n_bond))])
            mu_bl, sigma_bl = black_litterman_relative(
                pi, cov, P_full, q_vec, sigma_vec, TAU
            )

        for p, prof in PROFILES.items():
            eq_list   = list(current_eq_univ)
            bond_list = [b for b in BOND_TICKERS if b in current_universe]
            w_new = mvo_optimize_unified(
                mu_bl, sigma_bl, prof['gamma'],
                equity_tickers    = eq_list,
                bond_tickers      = bond_list,
                max_equity_weight = prof['max_equity_weight'],
                max_sector        = prof['max_sector'],
                sector_map        = current_sector_map,
                max_bond_total    = prof['max_bond_total'],
                max_single_bond   = prof['max_single_bond'],
            )
            current_weights[p] = w_new

print('\n백테스트 완료!')


In [ ]:
def compute_metrics(values, dates_list):
    vals = np.array(values)
    rets = np.diff(vals) / vals[:-1]
    n_years = (pd.Timestamp(dates_list[-1]) - pd.Timestamp(dates_list[0])).days / 365.25
    total_ret = vals[-1] / vals[0] - 1
    ann_ret = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
    ann_vol = rets.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    running_max = np.maximum.accumulate(vals)
    mdd = ((vals - running_max) / running_max).min()
    down = rets[rets < 0]
    sortino = ann_ret / (down.std() * np.sqrt(252)) if len(down) > 0 else 0
    calmar = ann_ret / abs(mdd) if mdd != 0 else 0
    return {'ann_return': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe,
            'mdd': mdd, 'sortino': sortino, 'calmar': calmar}


dates = list(analysis_days)
metrics = {}
for p in PROFILES:
    metrics[f'RelView_{p}'] = compute_metrics(port_values[p], dates)

with open(CACHE / 'spy_prices.pkl', 'rb') as f:
    spy_raw = pickle.load(f)
spy_aligned = spy_raw.loc[ANALYSIS_START:]
spy_norm = (spy_aligned / spy_aligned.iloc[0]).values
metrics['SPY_BnH'] = compute_metrics(spy_norm, list(spy_aligned.index))

print('=== 상대뷰 성능 요약 ===')
for name, m in metrics.items():
    print(f'{name:22s}: 연수익률={m["ann_return"]:>7.2%} | Sharpe={m["sharpe"]:>6.3f} | MDD={m["mdd"]:>7.2%}')

pd.DataFrame(metrics).T.to_csv(DATA / 'step5c_relview_metrics.csv')
print('저장: data/step5c_relview_metrics.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

COLORS = {
    'RelView_Aggressive': '#c0392b', 'RelView_Neutral': '#2980b9',
    'RelView_Conservative': '#27ae60', 'SPY_BnH': '#f39c12'
}

# 누적 수익률
ax = axes[0]
for p in PROFILES:
    ax.plot(dates, port_values[p], label=f'RelView_{p}',
            color=COLORS.get(f'RelView_{p}', 'gray'), linewidth=1.5)
ax.plot(list(spy_aligned.index), spy_norm, label='SPY B&H',
        color=COLORS['SPY_BnH'], linewidth=1.5, linestyle=':')
ax.set_title('Step5C: 상대뷰 BL 누적 수익률')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 성능 비교 바차트 (Sharpe)
ax2 = axes[1]
names = list(metrics.keys())
sharpe_vals = [metrics[n]['sharpe'] for n in names]
colors_bar = [COLORS.get(n, '#7f8c8d') for n in names]
bars = ax2.bar(names, sharpe_vals, color=colors_bar, edgecolor='white')
for bar, v in zip(bars, sharpe_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{v:.3f}', ha='center', va='bottom', fontsize=8)
ax2.set_title('Sharpe 비율 비교')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Step5C: 횡단면 랭크 기반 상대뷰 BL', fontsize=13)
plt.tight_layout()
plt.savefig(IMAGES / 'step5c_relview.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: images/step5c_relview.png')

## 셀 7: Step5 / Step5A / Step5C 통합 비교

3가지 전략의 성능 지표를 통합 비교


In [ ]:
# 이전 Step들의 결과 파일 로드 후 통합 비교
all_metrics = {}

for fname, prefix in [
    ('step5_metrics.csv',          'Base'),
    ('step5a_dualmom_metrics.csv',  'DualMom'),
    ('step5c_relview_metrics.csv',  'RelView'),
]:
    fpath = DATA / fname
    if fpath.exists():
        df = pd.read_csv(fpath, index_col=0)
        for idx, row in df.iterrows():
            if idx != 'SPY_BnH':
                all_metrics[f'{prefix}_{idx.replace(prefix+"_", "")}'] = row.to_dict()
        if 'SPY_BnH' in df.index and 'SPY_BnH' not in all_metrics:
            all_metrics['SPY_BnH'] = df.loc['SPY_BnH'].to_dict()

if all_metrics:
    compare_df = pd.DataFrame(all_metrics).T
    print('=== 전략별 통합 성능 비교 ===')
    print(f'{"전략":28s} {"연수익률":>10s} {"Sharpe":>8s} {"MDD":>10s}')
    print('-' * 60)
    for name, row in compare_df.iterrows():
        print(f'{name:28s} {row["ann_return"]:>10.2%} {row["sharpe"]:>8.3f} {row["mdd"]:>10.2%}')
    compare_df.to_csv(DATA / 'step5_all_comparison.csv')
    print('\n저장: data/step5_all_comparison.csv')
else:
    print('다른 Step5 노트북을 먼저 실행하세요')
